In [1]:
from sklearn.model_selection import cross_val_score

In [2]:
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.svm import SVC

In [3]:
# Import necessary libraries
import optuna
from sklearn.datasets import load_diabetes
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

# Load the Pima Indian Diabetes dataset from sklearn
# Note: Scikit-learn's built-in 'load_diabetes' is a regression dataset.
# We will load the actual diabetes dataset from an external source
import pandas as pd

# Load the Pima Indian Diabetes dataset (from UCI repository)
url = "https://raw.githubusercontent.com/jbrownlee/Datasets/master/pima-indians-diabetes.data.csv"
columns = ['Pregnancies', 'Glucose', 'BloodPressure', 'SkinThickness', 'Insulin', 'BMI',
           'DiabetesPedigreeFunction', 'Age', 'Outcome']

# Load the dataset
df = pd.read_csv(url, names=columns)

df.head()

/home/roben/Codes/PracticalDeepLearning/practicaldeeplearning/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


,Pregnancies,Glucose,BloodPressure,SkinThickness,Insulin,BMI,DiabetesPedigreeFunction,Age,Outcome
0,6,148,72,35,0,33.6,0.627,50,1
1,1,85,66,29,0,26.6,0.351,31,0
2,8,183,64,0,0,23.3,0.672,32,1
3,1,89,66,23,94,28.1,0.167,21,0
4,0,137,40,35,168,43.1,2.288,33,1


In [4]:
import numpy as np

# Replace zero values with NaN in columns where zero is not a valid value
cols_with_missing_vals = ['Glucose', 'BloodPressure', 'SkinThickness', 'Insulin', 'BMI']
df[cols_with_missing_vals] = df[cols_with_missing_vals].replace(0, np.nan)

# Impute the missing values with the mean of the respective column
df.fillna(df.mean(), inplace=True)

# Check if there are any remaining missing values
print(df.isnull().sum())

Pregnancies                 0
Glucose                     0
BloodPressure               0
SkinThickness               0
Insulin                     0
BMI                         0
DiabetesPedigreeFunction    0
Age                         0
Outcome                     0
dtype: int64


In [5]:
# Split into features (X) and target (y)
X = df.drop('Outcome', axis=1)
y = df['Outcome']

# Split data into training and test sets (70% train, 30% test)
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.3, random_state=42)

# Optional: Scale the data for better model performance
scaler = StandardScaler()
X_train = scaler.fit_transform(X_train)
X_test = scaler.transform(X_test)

# Check the shape of the data
print(f'Training set shape: {X_train.shape}')
print(f'Test set shape: {X_test.shape}')

Training set shape: (537, 8)
Test set shape: (231, 8)


In [12]:
def objective(trial):
    classifier_name = trial.suggest_categorical('classifier', ['SVM', 'RandomForest', 'GradientBoosting'])

    if classifier_name == 'SVM':
        c = trial.suggest_float('C', 0.1, 100, log=True)
        kernel = trial.suggest_categorical('kernel', ['linear', 'rbf', 'poly', 'sigmoid'])
        gamma = trial.suggest_categorical('gamma', ['scale', 'auto'])
        
        model = SVC(C=c, kernel=kernel, gamma=gamma, random_state=42)

    elif classifier_name == 'RandomForest':
        n_estimators = trial.suggest_int('n_estimators', 50, 300)
        max_depth = trial.suggest_int('max_depth', 3, 20)
        min_samples_split = trial.suggest_int('min_samples_split', 2, 10)
        min_samples_leaf = trial.suggest_int('min_samples_leaf', 1, 10)
        bootstrap = trial.suggest_categorical('bootstrap', [True, False])

        model = RandomForestClassifier(
            n_estimators=n_estimators,
            max_depth=max_depth,
            min_samples_split=min_samples_split,
            min_samples_leaf=min_samples_leaf,
            bootstrap=bootstrap,
            random_state=42
        )

    elif classifier_name == 'GradientBoosting':
        n_estimators = trial.suggest_int('n_estimators', 50, 300)
        learning_rate = trial.suggest_int('learning_rate', 0.01, 0.3, log=False)
        max_depth = trial.suggest_int('max_depth', 3, 20)
        min_samples_split = trial.suggest_int('min_samples_split', 2, 10)
        min_samples_leaf = trial.suggest_int('min_samples_leaf', 1, 10)

        model = GradientBoostingClassifier(
            n_estimators=n_estimators,
            learning_rate=learning_rate,
            max_depth=max_depth,
            min_samples_split=min_samples_split,
            min_samples_leaf=min_samples_leaf,
            random_state=42
        )

    score = cross_val_score(model, X_train, y_train, cv=3, scoring='accuracy').mean()
    return score

In [13]:
# Create a study and optimize it using CmaEsSampler
study = optuna.create_study(direction='maximize')
study.optimize(objective, n_trials=100)

[I 2026-06-24 18:37:49,810] A new study created in memory with name: no-name-1f09a468-76b1-424e-a30f-a96f5ea936da
[I 2026-06-24 18:37:49,846] Trial 0 finished with value: 0.7076350093109869 and parameters: {'classifier': 'SVM', 'C': 17.352258986362866, 'kernel': 'poly', 'gamma': 'auto'}. Best is trial 0 with value: 0.7076350093109869.
[I 2026-06-24 18:37:50,584] Trial 1 finished with value: 0.6499068901303539 and parameters: {'classifier': 'GradientBoosting', 'n_estimators': 177, 'learning_rate': 0, 'max_depth': 4, 'min_samples_split': 8, 'min_samples_leaf': 3}. Best is trial 0 with value: 0.7076350093109869.
[I 2026-06-24 18:37:50,624] Trial 2 finished with value: 0.7113594040968342 and parameters: {'classifier': 'SVM', 'C': 35.11063513259725, 'kernel': 'poly', 'gamma': 'auto'}. Best is trial 2 with value: 0.7113594040968342.
[I 2026-06-24 18:37:51,261] Trial 3 finished with value: 0.6499068901303539 and parameters: {'classifier': 'GradientBoosting', 'n_estimators': 66, 'learning_rate

In [14]:
# Retrieve the best trial
best_trial = study.best_trial
print(f"Best trial parameters: ", best_trial.params)
print(f"Best trial accuracy: ", best_trial.value)

Best trial parameters:  {'classifier': 'SVM', 'C': 0.1128866446123798, 'kernel': 'linear', 'gamma': 'scale'}
Best trial accuracy:  0.7895716945996275


In [15]:
study.trials_dataframe()

,number,value,datetime_start,datetime_complete,duration,params_C,params_bootstrap,params_classifier,params_gamma,params_kernel,params_learning_rate,params_max_depth,params_min_samples_leaf,params_min_samples_split,params_n_estimators,state
0,0,0.707635,2026-06-24 18:37:49.812181,2026-06-24 18:37:49.846214,0 days 00:00:00.034033,17.352259,NaN,SVM,auto,poly,NaN,NaN,NaN,NaN,NaN,COMPLETE
1,1,0.649907,2026-06-24 18:37:49.847021,2026-06-24 18:37:50.584810,0 days 00:00:00.737789,NaN,NaN,GradientBoosting,NaN,NaN,0.0,4.0,3.0,8.0,177.0,COMPLETE
2,2,0.711359,2026-06-24 18:37:50.585593,2026-06-24 18:37:50.624649,0 days 00:00:00.039056,35.110635,NaN,SVM,auto,poly,NaN,NaN,NaN,NaN,NaN,COMPLETE
3,3,0.649907,2026-06-24 18:37:50.625565,2026-06-24 18:37:51.261301,0 days 00:00:00.635736,NaN,NaN,GradientBoosting,NaN,NaN,0.0,16.0,1.0,3.0,66.0,COMPLETE
4,4,0.763501,2026-06-24 18:37:51.262029,2026-06-24 18:37:51.285696,0 days 00:00:00.023667,0.236574,NaN,SVM,scale,sigmoid,NaN,NaN,NaN,NaN,NaN,COMPLETE
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
95,95,0.757914,2026-06-24 18:38:05.609085,2026-06-24 18:38:06.490344,0 days 00:00:00.881259,NaN,True,RandomForest,NaN,NaN,NaN,18.0,7.0,4.0,271.0,COMPLETE
96,96,0.787709,2026-06-24 18:38:06.491426,2026-06-24 18:38:06.512397,0 days 00:00:00.020971,0.104035,NaN,SVM,scale,linear,NaN,NaN,NaN,NaN,NaN,COMPLETE
97,97,0.649907,2026-06-24 18:38:06.513233,2026-06-24 18:38:08.159746,0 days 00:00:01.646513,NaN,NaN,GradientBoosting,NaN,NaN,0.0,11.0,2.0,7.0,219.0,COMPLETE
98,98,0.789572,2026-06-24 18:38:08.160757,2026-06-24 18:38:08.180251,0 days 00:00:00.019494,0.149488,NaN,SVM,scale,linear,NaN,NaN,NaN,NaN,NaN,COMPLETE


In [16]:
study.trials_dataframe()['params_classifier'].value_counts()

params_classifier
SVM                 79
GradientBoosting    11
RandomForest        10
Name: count, dtype: int64

In [18]:
study.trials_dataframe().groupby('params_classifier')['value'].mean()

params_classifier
GradientBoosting    0.649907
RandomForest        0.769460
SVM                 0.775735
Name: value, dtype: float64